In [1]:
import re
import pandas as pd
import requests
import yaml
import time
import random
import os

In [ ]:
# Load credentials from creds.yaml
with open("creds.yaml", "r") as stream:
    try:
        creds = yaml.safe_load(stream)
    except yaml.YAMLError as e:
        print(f"Error loading YAML file: {e}")

# initialize SocialCrawl API key to use for requests
socialcrawl_key = creds["socialcrawl"]["api_key"]

# Used to test the SocialCrawl API for Instagram profile data
res = requests.get(
    "https://www.socialcrawl.dev/v1/instagram/profile",
    params={"handle": "instagram"},
    headers={"x-api-key": socialcrawl_key},
    timeout=15,
)

# Place the SocialCrawl API for Facebook profile data here (to do..)

# Make sure the request was successful
print(res.status_code)
print(res.text[:500])

200
{"success":true,"platform":"instagram","endpoint":"/v1/instagram/profile","data":{"author":{"id":"25025320","username":"instagram","display_name":"Instagram","avatar_url":"https://scontent-dfw5-2.cdninstagram.com/v/t51.82787-19/550891366_18667771684001321_1383210656577177067_n.jpg?_nc_cat=1&ig_cache_key=GGbv1SApFvlLPVJCAOtF0XrDJjITbmNDAQAB0j-ccb7-5&ccb=7-5&_nc_sid=669407&efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLnd3dy4xMDgwLkMzIn0%3D&_nc_ohc=JMCO63q6phEQ7kNvwHxZWee&_nc_oc=Adr6xM-1WRgn6rpasqRpeA2a


In [3]:
# Downloading recent posts (SocialCrawl gives 100 credits TOTAL)
def get_posts(handle, max_pages=3):
    """ 
    Helper function to get posts from an Instagram account using the SocialCrawl API.
    
    Args:
        handle (str): Instagram username to scrape posts from.
        max_pages (int): Maximum number of pages to scrape. Each page contains up to 12 posts. Default is 3 pages (36 posts).
    Returns:
        list: A list of posts retrieved from the Instagram account.
    """
    rows, cursor = [], None
    for _ in range(max_pages):
        params = {"handle": handle}
        if cursor:
            params["next_max_id"] = cursor
        res = requests.get(
            "https://www.socialcrawl.dev/v1/instagram/profile/posts",
            params=params, headers={"x-api-key": socialcrawl_key}, timeout=30,
        )
        res.raise_for_status()
        body = res.json()
        data = body["data"]
        if len(rows) == 0:
            print(body.get("credits_remaining"))
            print(str(data)[:1500])
        items = data if isinstance(data, list) else (data.get("posts") or data.get("items") or [])
        rows.extend(items)
        cursor = (data.get("next_max_id") if isinstance(data, dict) else None) or body.get("next_max_id")
        if not cursor:
            break
        time.sleep(random.uniform(1, 3))
    return rows


In [4]:
# Import club and university information from the spreadsheet we created
social_media = pd.read_csv("Data/Social Media for University - Sheet1.csv")
social_media.columns = social_media.columns.str.strip()

for column in ["university", "club", "instagram_handle"]:
    social_media[column] = social_media[column].fillna("").astype(str).str.strip()

# Focusing on instagram pages first:
instagram_rows = social_media[
    social_media["instagram_handle"].ne("")
    & social_media["Instagram/Facebook link"].fillna("").str.contains("instagram.com", case=False, regex=False)
]

club = dict(zip(instagram_rows["instagram_handle"], instagram_rows["club"]))
university = dict(zip(instagram_rows["instagram_handle"], instagram_rows["university"]))
instagram_handle = instagram_rows["instagram_handle"].drop_duplicates().tolist()

print("Instagram handles loaded:", instagram_handle)

raw_by_handle = {
    handle: get_posts(handle)
    for handle in instagram_handle
}

Instagram handles loaded: ['ju.ieee', 'diu.cpc', 'diu_robotics_club', 'aiubfinanceclub', 'aiubenglish.club', 'aiubcomputerclub', 'aiubpcofficial', 'aiubfilmclub_official', 'aiubesportsclub']
88
{'items': [{'post': {'id': '3208589558413751241', 'url': 'https://www.instagram.com/p/CyHMzQbR3PJ/', 'content': {'text': None, 'media_urls': 'https://scontent-lga3-1.cdninstagram.com/v/t51.82787-15/628065112_18360617242164744_448216875388839578_n.jpg?stp=dst-jpg_e35_tt6&_nc_cat=110&_nc_map=urlgen_bucketless&ig_cache_key=MzIwODU4OTU1ODQxMzc1MTI0MQ%3D%3D.3-ccb7-5&ccb=7-5&_nc_sid=58cdad&efg=eyJ2ZW5jb2RlX3RhZyI6IkZFRUQueHBpZHMuMzk5LnNkci5yZWd1bGFyX3Bob3RvLkMzIn0%3D&_nc_ohc=QBROhrPaRw8Q7kNvwFkUbTg&_nc_oc=AdoxoXndt-m5jWLLhSbWW7Gjo78IY44HlYKfP70lDXt-avuyhqcyeO2AbEMiV-xAZsM&_nc_ad=z-m&_nc_cid=0&_nc_zt=23&_nc_ht=scontent-lga3-1.cdninstagram.com&_nc_gid=y2WlwLcEP6U6woIJZTxaQw&_nc_ss=7339b&_nc_ct=RQosPBUIFQIlBAAAAA&oh=00_AQRwYKFfsLWcCLSgN2E6yrfBGGw_CgFfWgLnLma_ErEd0A&oe=6B068E34', 'thumbnail_url': None, 'd

In [5]:
# Dictionaries: words/phrases that signal each concept (classifier)
ai_terms = [
    "ai", "a.i.", "artificial intelligence", "machine learning",
    "generative ai", "genai", "chatgpt", "gemini", "claude", "llm",
    "deep learning", "neural network", "data science", "prompt engineering",
]
event_terms = [
    "event", "seminar", "workshop", "conference", "symposium", "webinar",
    "talk", "meetup", "bootcamp", "hackathon", "summit", "masterclass",
    "competition", "register", "join us",
]

In [6]:
def make_pattern(terms):
    """Match whole words only, so 'ai' does NOT match 'said', 'maintain', 'email'."""
    return re.compile(r"\b(" + "|".join(re.escape(t) for t in terms) + r")\b", re.IGNORECASE)

ai_pattern = make_pattern(ai_terms)
event_pattern = make_pattern(event_terms)

In [7]:
def classify_post(caption, instagram_handle=""):
    text = re.sub(r"\s+", " ", str(caption or ""))
    ai_hits = sorted({m.lower() for m in ai_pattern.findall(text)})
    event_hits = sorted({m.lower() for m in event_pattern.findall(text)})
    return {
        "university": university.get(instagram_handle, university),
        "club_name": club.get(instagram_handle, instagram_handle),
        "instagram_handle": instagram_handle,
        "is_ai": len(ai_hits) > 0,
        "is_event": len(event_hits) > 0,
        "ai_event": len(ai_hits) > 0 and len(event_hits) > 0,
        "ai_matches": ai_hits,
        "event_matches": event_hits,
    }

In [ ]:
def find_caption(item):
    post = item.get("post", item)
    content = post.get("content") or {}
    return content.get("text") or ""

rows = []
for handle, raw in raw_by_handle.items():
    for item in raw:
        rows.append({
            "url": item.get("post", item).get("url"),
            "caption": find_caption(item),
            "instagram_handle": handle,
        })

posts = pd.DataFrame(rows)
classified = posts.apply(
    lambda row: classify_post(row["caption"], row["instagram_handle"]),
    axis=1,
).apply(pd.Series)
posts = pd.concat([posts, classified], axis=1)

OUT = "Data/bangladesh_university_ai_promotion_classified.csv"

posts["club"] = posts["club_name"]

# doing this to avoid duplicates if the script is run multiple times
if os.path.exists(OUT):
    posts = posts[~posts.url.isin(pd.read_csv(OUT).url)]

108 posts | 105 with captions | 11 AI | 44 event | 9 AI event


In [10]:
# appending new posts to the csv
posts.to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
print(len(posts), "new posts added")
posts.head(10)

108 new posts added


,url,caption,instagram_handle,university,club_name,instagram_handle,is_ai,is_event,ai_event,ai_matches,event_matches,club
0,https://www.instagram.com/p/CyHMzQbR3PJ/,,ju.ieee,Jahangirnagar University,IEEE JU Student Branch,ju.ieee,False,False,False,[],[],IEEE JU Student Branch
1,https://www.instagram.com/p/CyHMyIux8VG/,,ju.ieee,Jahangirnagar University,IEEE JU Student Branch,ju.ieee,False,False,False,[],[],IEEE JU Student Branch
2,https://www.instagram.com/p/CyHMxECR0tU/,,ju.ieee,Jahangirnagar University,IEEE JU Student Branch,ju.ieee,False,False,False,[],[],IEEE JU Student Branch
3,https://www.instagram.com/p/DdrbJCeoCdj/,Come join us for our Soldering Workshop!!! Don...,ju.ieee,Jahangirnagar University,IEEE JU Student Branch,ju.ieee,False,True,False,[],"[join us, workshop]",IEEE JU Student Branch
4,https://www.instagram.com/p/DW6_72GEXps/,⚡ REMINDER: IEEE Soldering Masterclass is TOMO...,ju.ieee,Jahangirnagar University,IEEE JU Student Branch,ju.ieee,False,True,False,[],[masterclass],IEEE JU Student Branch
5,https://www.instagram.com/p/DWozem8jf3Y/,🔧⚡ IEEE JU Soldering Masterclass - Round 2! 🚀\...,ju.ieee,Jahangirnagar University,IEEE JU Student Branch,ju.ieee,False,True,False,[],[masterclass],IEEE JU Student Branch
6,https://www.instagram.com/p/DV04EZ9DrSY/,🚀 IEEE JU Officer Applications EXTENDED! ⚡\n\n...,ju.ieee,Jahangirnagar University,IEEE JU Student Branch,ju.ieee,False,False,False,[],[],IEEE JU Student Branch
7,https://www.instagram.com/p/DVPdrr1DaqB/,🚀 IEEE JU Officer Applications Are NOW OPEN! ⚡...,ju.ieee,Jahangirnagar University,IEEE JU Student Branch,ju.ieee,False,False,False,[],[],IEEE JU Student Branch
8,https://www.instagram.com/p/DT7DvM4Dfb9/,🔧 Join us for the IEEE JU Soldering Masterclas...,ju.ieee,Jahangirnagar University,IEEE JU Student Branch,ju.ieee,False,True,False,[],"[join us, masterclass, workshop]",IEEE JU Student Branch
9,https://www.instagram.com/p/DTrp3mNjTOs/,🚨 ALL NEW IEEE JU DISCORD SERVER IS LIVE! 🚨\n\...,ju.ieee,Jahangirnagar University,IEEE JU Student Branch,ju.ieee,False,False,False,[],[],IEEE JU Student Branch
